# Stock Price Predictor — Model Notebook
This notebook trains and evaluates both Linear Regression and LSTM models on real stock data.

In [ ]:
import sys
sys.path.append('../backend')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

from models.linear_model import LinearRegressionModel
from models.lstm_model import LSTMModel
from utils.metrics import compute_metrics

print('All imports successful!')

## 1. Load Stock Data

In [ ]:
TICKER = 'AAPL'
stock = yf.Ticker(TICKER)
df = stock.history(period='2y')
df.index = df.index.tz_localize(None)
close = df['Close'].values

print(f'Loaded {len(close)} days of {TICKER} closing prices')
print(f'Date range: {df.index[0].date()} to {df.index[-1].date()}')
print(f'Price range: ${close.min():.2f} - ${close.max():.2f}')
df[['Close']].tail()

## 2. Visualize Raw Data

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(df.index, close, color='steelblue', linewidth=1.5, label='Close Price')
plt.title(f'{TICKER} — 2 Year Closing Price History', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.tight_layout()
plt.savefig(f'../backend/screenshots/{TICKER}_history.png', dpi=150)
plt.show()

## 3. Train Linear Regression Model

In [ ]:
lr_model = LinearRegressionModel(lookback=30)
lr_model.train(close)
lr_preds, lr_future = lr_model.predict(close, forecast_days=30)

actual_for_eval = close[30:]
lr_metrics = compute_metrics(actual_for_eval, lr_preds)
print('Linear Regression Metrics:')
for k, v in lr_metrics.items():
    print(f'  {k}: {v}')

## 4. Train LSTM Model

In [ ]:
lstm_model = LSTMModel(lookback=60, epochs=30)
lstm_model.train(close)
lstm_preds, lstm_future = lstm_model.predict(close, forecast_days=30)

actual_for_eval_lstm = close[60:]
lstm_metrics = compute_metrics(actual_for_eval_lstm, lstm_preds)
print('LSTM Metrics:')
for k, v in lstm_metrics.items():
    print(f'  {k}: {v}')

## 5. Compare Predictions vs Actual

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Linear Regression
axes[0].plot(actual_for_eval[-100:], label='Actual', color='steelblue')
axes[0].plot(lr_preds[-100:], label='Predicted', color='orange', linestyle='--')
axes[0].set_title(f'Linear Regression — {TICKER} (last 100 days)')
axes[0].legend()
axes[0].set_ylabel('Price (USD)')

# LSTM
axes[1].plot(actual_for_eval_lstm[-100:], label='Actual', color='steelblue')
axes[1].plot(lstm_preds[-100:], label='Predicted', color='tomato', linestyle='--')
axes[1].set_title(f'LSTM — {TICKER} (last 100 days)')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'../backend/screenshots/{TICKER}_predictions.png', dpi=150)
plt.show()

## 6. 30-Day Future Forecast

In [ ]:
future_dates = pd.date_range(start=df.index[-1] + pd.Timedelta(days=1), periods=30, freq='B')

plt.figure(figsize=(12, 5))
plt.plot(df.index[-60:], close[-60:], label='Historical', color='steelblue')
plt.plot(future_dates, lr_future, label='LR Forecast', color='orange', linestyle='--', marker='o', markersize=3)
plt.plot(future_dates, lstm_future, label='LSTM Forecast', color='tomato', linestyle='--', marker='o', markersize=3)
plt.axvline(x=df.index[-1], color='gray', linestyle=':', alpha=0.7, label='Forecast Start')
plt.title(f'{TICKER} — 30-Day Future Price Forecast', fontsize=13)
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.tight_layout()
plt.savefig(f'../backend/screenshots/{TICKER}_forecast.png', dpi=150)
plt.show()

print(f'\nLast known price:  ${close[-1]:.2f}')
print(f'LR 30-day forecast:   ${lr_future[-1]:.2f}')
print(f'LSTM 30-day forecast: ${lstm_future[-1]:.2f}')

## 7. Metrics Summary Table

In [ ]:
summary = pd.DataFrame([lr_metrics, lstm_metrics], index=['Linear Regression', 'LSTM'])
print(summary.to_string())